# Week 6 lab · CO₂ emissions and renewable energy projects

**CM3801 · CMT309/509** · two-hour lab

You will use this week's pandas on two real datasets:
* **Part A · Asking questions.** CO₂ emissions per country (Our World in Data): summary statistics, groups, top-10 tables and missing values.
* **Part B · Cleaning a real file.** The UK's Renewable Energy Planning Database, exactly as the government publishes it. You will find and fix its problems yourself, then answer questions with `groupby`.

**How this lab works**
* Work through the tasks in order. Each task has an empty cell for your code, followed by a **checkpoint** cell.
* Run the checkpoint after your code. If it prints `✔`, move on. If it raises an `AssertionError`, read the message, fix your code and try again.
* **Question** cells ask you to interpret a result in a sentence or two. Write your answer in the text cell (double-click to edit).
* Part B starts with a fresh dataset: if you are short of time in Part A, move on to Part B after Task A3.
* The **Challenge** at the end is optional.

## Setup

Run this cell first.

*Data: Our World in Data, CO₂ and Greenhouse Gas Emissions dataset (CC BY 4.0), a cleaned copy. Renewable Energy Planning Database (REPD), Q2 July 2025, Department for Energy Security and Net Zero: contains public sector information licensed under the Open Government Licence v3.0.*

In [ ]:
# --- Setup: run this cell first -------------------------------------------------
# Course datasets live in the course repository under 2026-27/datasets/.
# get_data() uses a local copy if the notebook runs inside a copy of the repository;
# otherwise (e.g. in Google Colab) it downloads the file once and reuses it.
import os
import urllib.request

COURSE_REPO = "OWNER/REPO"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/main/2026-27/"


def course_file(relpath):
    """Return a local path to a course file such as 'datasets/owid_co2_cleaned.csv'."""
    for root in ("../../..", "."):                     # 1) running inside a copy of the repository
        path = os.path.join(root, relpath)
        if os.path.exists(path):
            return path
    path = os.path.join("course_files", relpath)       # 2) Colab: download once
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + relpath, path)
    return path


def get_data(name):
    """Return a local path to a course dataset, e.g. get_data('owid_co2_cleaned.csv')."""
    return course_file("datasets/" + name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Part A · Asking questions of the CO₂ table

The table has one row per country **and year**. It also contains rows for **groups** of countries (the world, continents, income groups) and for international transport. The cell below removes those groups for you: `isin(list)` asks "is the name in this list?", and `~` means "not", as with numpy masks. Then it keeps the year **2022**.

Units: `co2` in million tonnes per year, `co2_per_capita` in tonnes per person, `primary_energy_consumption` in terawatt-hours (TWh).

In [ ]:
co2 = pd.read_csv(get_data("owid_co2_cleaned.csv"))
GROUPS = [
    'Africa', 'Africa (GCP)', 'Asia', 'Asia (GCP)',
    'Asia (excl. China and India)', 'Central America (GCP)', 'Europe', 'Europe (GCP)',
    'Europe (excl. EU-27)', 'Europe (excl. EU-28)', 'European Union (27)', 'European Union (28)',
    'High-income countries', 'International aviation', 'International shipping', 'International transport',
    'Kuwaiti Oil Fires', 'Kuwaiti Oil Fires (GCP)', 'Least developed countries (Jones et al.)', 'Low-income countries',
    'Lower-middle-income countries', 'Middle East (GCP)', 'Non-OECD (GCP)', 'North America',
    'North America (GCP)', 'North America (excl. USA)', 'OECD (GCP)', 'OECD (Jones et al.)',
    'Oceania', 'Oceania (GCP)', 'Ryukyu Islands', 'Ryukyu Islands (GCP)',
    'South America', 'South America (GCP)', 'Upper-middle-income countries', 'World',
]
countries = co2[~co2["country"].isin(GROUPS)]
c22 = countries[countries["year"] == 2022]
print(co2.shape, countries.shape, c22.shape)
c22.head()

### Example (worked for you)

*Among the countries with more than 1 million people, which had the highest CO₂ emissions per person in 2022?* Read the code, run it, and make sure you understand each line: you'll use the same pattern below.

In [ ]:
big = c22[c22["population"] > 1_000_000]
top = big.sort_values("co2_per_capita", ascending=False).head(1)
name = top["country"].iloc[0]
value = top["co2_per_capita"].iloc[0]
print(f"Among countries with more than 1 million people, {name} emitted the most CO₂ per person in 2022: {value:.1f} t.")

## Task A1 · Summary statistics with a maximum

For `co2_per_capita` in 2022 (all countries in `c22`):
1. Drop the countries where it is missing, and store the result in `pc`.
2. Compute the mean, the standard deviation and the maximum, and find **which country** has the maximum.
3. Print one sentence like this one (your numbers will be filled in by the f-string):

```
In 2022 the average country emitted 4.58 ± 5.35 t of CO₂ per person; the maximum was 36.70 t (Qatar).
```

In [ ]:
# STUDENT TASK
# Task: pc (no missing co2_per_capita), mean_pc, std_pc, max_pc and max_country, then one printed sentence
# Hint: .mean(), .std(), .max(); for the country, sort or filter on the maximum value
# your code here


In [ ]:
# Checkpoint A1
assert len(pc) == 214, "pc: drop the 5 countries with a missing co2_per_capita (219 - 5 = 214)"
assert round(mean_pc, 2) == 4.58, "mean_pc should be about 4.58"
assert round(std_pc, 2) == 5.35, "std_pc should be about 5.35 (pandas .std())"
assert max_country == "Qatar" and round(max_pc, 2) == 36.70, "the maximum is Qatar's 36.70 t"
print("✔ Task A1")

**Question A1.** The standard deviation (5.35 t) is larger than the mean (4.58 t). What does that tell you about how emissions per person are spread across countries? Is the mean a good summary of a "typical" country here?

*Your answer:*

## Task A2 · Groups: the UK decade by decade

How have the UK's emissions per person changed since 1900?
1. From `co2` keep the rows of the `"United Kingdom"` from 1900 onwards, as `uk` (add `.copy()` at the end so that you can add a column).
2. Add a column `decade`: the year rounded down to its decade (1994 → 1990). *Hint:* integer division `//` from Week 1.
3. Group by `decade` and compute the **mean** `co2_per_capita` per decade, as `by_decade`.
4. Find the decade with the **highest** and the decade with the **lowest** mean, and print them.

In [ ]:
# STUDENT TASK
# Task: uk (1900 onwards, copy), uk["decade"], by_decade (groupby mean), then highest and lowest decade
# Hint: uk.groupby("decade")["co2_per_capita"].mean(); sort_values() puts the lowest first
# your code here


In [ ]:
# Checkpoint A2
assert "decade" in uk.columns and uk["decade"].min() == 1900, "uk['decade']: (year // 10) * 10, from 1900"
assert len(by_decade) == 13, "by_decade: one value per decade, 1900s to 2020s"
assert by_decade.idxmax() == 1970 and by_decade.idxmin() == 2020, "highest: the 1970s; lowest: the 2020s"
print("✔ Task A2")

**Question A2.** The 2020s value is an average over only four years (2020–2023). Why could that make the comparison with earlier decades unfair, and in which direction?

*Your answer:*

## Task A3 · The ten largest emitters

1. Store in `top10` the **ten countries with the highest total `co2`** in 2022, largest first, with only the columns `country` and `co2`.
2. Run the checkpoint: it also **draws** your table as a bar chart (plotting is next week's topic; the code is given).

In [ ]:
# STUDENT TASK
# Task: top10 = the 10 largest co2 values in c22, largest first, columns country and co2
# Hint: sort_values(..., ascending=False), .head(10), then a list of columns
# your code here


In [ ]:
# Checkpoint A3 (draws your top10 as a chart)
assert list(top10.columns) == ["country", "co2"], "top10: keep only the columns country and co2"
assert top10["country"].tolist()[:3] == ["China", "United States", "India"], "sort by co2, largest first"
assert len(top10) == 10 and top10["country"].iloc[-1] == "South Korea", "keep exactly ten rows"
ax = top10.plot.barh(x="country", y="co2", legend=False, figsize=(6, 4), color="#4a7bb7")
ax.invert_yaxis()
ax.set_xlabel("CO₂ emissions in 2022 (million tonnes)")
plt.show()
print("✔ Task A3")

**Question A3.** China emits more than twice as much CO₂ as the United States. Does that mean the average person in China emits more than the average person in the US? Which column of `c22` would you use to check, and what does it show?

*Your answer:*

## Task A4 · Missing values: drop or fill?

`primary_energy_consumption` is missing for many countries in 2022: in fact, for most of them.
1. Store the number of missing values in `n_missing`.
2. Compute the mean of the column in two ways: `mean_drop` **ignores** the missing values (pandas' `.mean()` does this by default, like `dropna`), and `mean_fill0` first **fills them with 0** (`fillna(0)`).

In [ ]:
# STUDENT TASK
# Task: n_missing, mean_drop and mean_fill0 for c22["primary_energy_consumption"]
# Hint: .isna().sum(); .mean(); .fillna(0).mean()
# your code here


In [ ]:
# Checkpoint A4
assert n_missing == 140, "n_missing: count the NaN values with isna().sum()"
assert round(mean_drop) == 2061, "mean_drop: the mean of the values that are present (about 2,061 TWh)"
assert round(mean_fill0) == 744, "mean_fill0: fill with 0 first, then take the mean (about 744 TWh)"
print("✔ Task A4")

**Question A4.** The two means differ by a factor of almost three. Which one is a sensible answer to "how much energy does a country use on average?", and why is `fillna(0)` wrong here? Can you think of a column where filling with 0 *would* be right?

*Your answer:*

# Part B · Cleaning a real government file

The **Renewable Energy Planning Database (REPD)** lists every renewable electricity project in the UK above a small size, from planning application to operation: solar farms, wind farms, batteries and more. We use the file exactly as the government publishes it, and it has the problems real files have. Your job is to find them and fix them, using only Week 6 tools.

## Task B1 · Load the file

**Before you start:** run the cell below as it is. It fails on purpose: read the **last line** of the error message.

In [ ]:
# ⚠️ This cell raises an error on purpose
raw = pd.read_csv(get_data("repd_raw_q2_jul_2025.csv"))

The file is not saved as UTF-8. It was exported from Excel in the Windows encoding, **`cp1252`** (see the note in the lecture). Load it correctly into `raw` and print its shape.

In [ ]:
# STUDENT TASK
# Task: raw = the REPD file read with the right encoding
# Hint: pd.read_csv(..., encoding="cp1252")
# your code here


In [ ]:
# Checkpoint B1
assert raw.shape == (12977, 53), "raw should have 12,977 projects and 53 columns"
print("✔ Task B1")

## Task B2 · Keep five columns and tidy the text

53 columns are far more than we need.
1. Make `repd`: a **copy** of `raw` with only these five columns: `"Site Name"`, `"Technology Type"`, `"Installed Capacity (MWelec)"`, `"Development Status (short)"`, `"Country"`.
2. Look at `repd["Country"].value_counts()`. Something is wrong: find it, and fix the column with a `.str` method.
3. The status column has the same status spelled in different ways (look for `"Application WIthdrawn"`). Fix it so that every status is in *Title Case* and has no stray spaces.

In [ ]:
# STUDENT TASK
# Task: repd (five columns, a copy); clean Country; clean "Development Status (short)"
# Hint: .str.strip() removes spaces at both ends; .str.title() gives Title Case; methods can be chained
# your code here


In [ ]:
# Checkpoint B2
assert list(repd.columns) == ["Site Name", "Technology Type", "Installed Capacity (MWelec)",
                              "Development Status (short)", "Country"], "repd: exactly the five columns, in this order"
assert repd["Country"].nunique() == 4, "Country should have exactly 4 values (England, Scotland, Wales, Northern Ireland)"
assert repd["Development Status (short)"].nunique() == 15, "status: 15 distinct values after .str.strip().str.title()"
print("✔ Task B2")

**Question B2.** Before the fix, `value_counts()` listed `England` and `England ` as two different countries. Why does pandas treat them as different, and what would have gone wrong in a `groupby("Country")` if you had not noticed?

*Your answer:*

## Task B3 · Numbers stored as text

**Predict first:** run `repd.dtypes`. What dtype does the capacity column have, and why can't you add it up?

Then:
1. Create a new column `capacity_mw` from `"Installed Capacity (MWelec)"`: strip the spaces and convert to numbers, turning anything that isn't a number into `NaN`.
2. Store the **total** installed capacity of all projects in `total_mw`.

In [ ]:
# STUDENT TASK
# Task: repd["capacity_mw"] (float) and total_mw
# Hint: pd.to_numeric(repd[...].str.strip(), errors="coerce"), as in the lecture
# your code here


In [ ]:
# Checkpoint B3
assert repd["capacity_mw"].dtype == "float64", "capacity_mw: convert with pd.to_numeric(..., errors='coerce')"
assert repd["capacity_mw"].isna().sum() == 1136, "1,121 values were already missing and 15 were blank text: 1,136 NaN"
assert round(total_mw) == 430181, "total_mw: the sum of capacity_mw (about 430,181 MW)"
print("✔ Task B3")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
raw = pd.read_csv(get_data("repd_raw_q2_jul_2025.csv"), encoding="cp1252")
cols = ["Site Name", "Technology Type", "Installed Capacity (MWelec)", "Development Status (short)", "Country"]
repd = raw[cols].copy()
repd["Country"] = repd["Country"].str.strip()
repd["Development Status (short)"] = repd["Development Status (short)"].str.strip().str.title()
repd["capacity_mw"] = pd.to_numeric(repd["Installed Capacity (MWelec)"].str.strip(), errors="coerce")

**Question B3.** About 430,000 MW is several times the UK's whole electricity system. Look at `repd["Development Status (short)"].value_counts()`. Why is "the total capacity in the database" **not** the capacity that is generating electricity today?

*Your answer:*

## Task B4 · What is generating today?

1. Keep only the projects whose status is `"Operational"`, as `operational`.
2. Compute the total operational capacity **per technology**, largest first, as `by_tech` (a Series).
3. Print the top five.

In [ ]:
# STUDENT TASK
# Task: operational (filtered rows) and by_tech (capacity per technology, largest first)
# Hint: groupby("Technology Type")["capacity_mw"].sum().sort_values(ascending=False)
# your code here


In [ ]:
# Checkpoint B4
assert len(operational) == 3025, "operational: 3,025 projects have the status 'Operational'"
assert by_tech.index[:3].tolist() == ["Wind Onshore", "Wind Offshore", "Solar Photovoltaics"], \
    "by_tech: sum capacity_mw per Technology Type, largest first"
assert round(by_tech.iloc[0]) == 14738, "Wind Onshore: about 14,738 MW"
print("✔ Task B4")

**Question B4.** Onshore and offshore wind have almost the same operational capacity (about 14,700 MW each). Compare the **number of projects** behind each figure (`operational.groupby("Technology Type").size()`). What does that tell you about the size of a typical offshore wind farm?

*Your answer:*

## Task B5 · The four nations

1. Compute the total operational capacity per `Country`, largest first, as `by_country`.
2. Compute each nation's **share** of the UK total, as a percentage, as `share` (a Series), rounded to 1 decimal place.

In [ ]:
# STUDENT TASK
# Task: by_country (operational capacity per Country) and share (% of the UK total, 1 d.p.)
# Hint: a Series divided by a number divides every value: by_country / by_country.sum() * 100
# your code here


In [ ]:
# Checkpoint B5
assert by_country.index.tolist() == ["England", "Scotland", "Wales", "Northern Ireland"], "by_country: largest first"
assert share["Wales"] == 9.9 and share["England"] == 59.7, "share: by_country / by_country.sum() * 100, rounded to 1 d.p."
print("✔ Task B5")

**Question B5.** England has about 60 % of the operational capacity and Scotland about 28 %. Scotland has fewer than a tenth of the UK's people. Write two sentences for a reader of a newspaper about which nation does "most" for renewable electricity, including one caution about these figures.

*Your answer:*

## Challenge (optional) · Capacity per person with `merge`

A fair comparison divides by population. The populations below are approximate mid-2023 estimates (Office for National Statistics), in millions.

1. Turn `by_country` into a DataFrame with the columns `Country` and `capacity_mw` (*Hint:* `by_country.reset_index()`).
2. `merge` it with `nations` on `"Country"`.
3. Add a column `mw_per_million` and sort by it, largest first.

`merge` is in the *Optional* part of the lecture notebook: matching rows of two tables on a shared column.

In [ ]:
nations = pd.DataFrame({"Country": ["England", "Scotland", "Wales", "Northern Ireland"],
                        "population_m": [57.7, 5.5, 3.2, 1.9]})

In [ ]:
# STUDENT TASK
# Task: merge by_country with nations; mw_per_million; sort
# your code here


## Wrap-up

In Part A you asked questions of a clean table: summary statistics, groups, a ranking, and the difference between "missing" and "zero". In Part B you cleaned a real government file: the wrong encoding, stray spaces and inconsistent spelling in text columns, numbers stored as text, and a status column that changes what a total means. Most data work looks like Part B before it can look like Part A.

Next week you will turn tables like these into charts.